In [45]:
import os
import json
from IPython.display import Markdown
import gradio as gr
from openai import OpenAI
from scraper import fetch_website_contents
openai=OpenAI()

system_prompt=''' You are an helpful assistant and will be provided with website name and the url and you have to give a
beautiful brochure made from that ,a company brochure'''

def call_gpt4(prompt):
    message = [
        {'role': 'system', 'content': system_prompt},
        {'role': 'user', 'content': prompt},
    ]

    stream= openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=message,
        stream=True,
    )
    result = ""
    for chunk in stream:
        result += chunk.choices[0].delta.content or ""
        yield result
    

In [46]:
def call_gpt5(prompt):
    message = [
        {'role': 'system', 'content': system_prompt},
        {'role': 'user', 'content': prompt},
    ]

    stream = openai.chat.completions.create(
        model="gpt-5.1-mini",
        messages=message,
        stream=True,
    )
    result = ""
    for chunk in stream:
        result += chunk.choices[0].delta.content or ""
        yield result

In [47]:

def stream_brochure(company_name, url, model):
    yield ""
    prompt = f"Please generate a company brochure for {company_name}. Here is their landing page:\n"
    prompt += fetch_website_contents(url)
    if model=="gpt4":
        result = call_gpt4(prompt)
    elif model=="gpt5":
        result = call_gpt5(prompt)
    else:
        raise ValueError("Unknown model")
    yield from result

In [54]:
message_input=gr.Textbox(label="Company name:")
url_input=gr.Textbox(label="enter your url starts with https://")
model_selector=gr.Dropdown(["gpt4","gpt5"],label="select your model",value="gpt4")
message_output=gr.Markdown(label="RESPONSE:")

gr.Interface(fn=stream_brochure,title="brochure Generator(For Doro)",inputs=[message_input,url_input,model_selector],outputs=[message_output],examples=[
            ["Hugging Face", "https://huggingface.co", "gpt4"],
            ["Edward Donner", "https://edwarddonner.com", "gpt5"]
        ], flagging_mode="never").launch(share=True,auth=('ashmit', 'ash121'))


* Running on local URL:  http://127.0.0.1:7878
* Running on public URL: https://20af008b8a619f6a2f.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
